# Statistical analysis: culture, framing, language effects on unsafe/refused

`observations.csv` (loaded from `combined/`, the same data `load_combined_results.ipynb`
reads) gives binary verdicts per prompt — `unsafe` (0/1) and `refused` (0/1) — not a
continuous score. **ANOVA and box plots don't directly apply to a binary outcome**;
this notebook uses the binary-data analogs instead:

| Continuous-data tool | Binary-data analog (used here) |
|---|---|
| t-test / ANOVA | Chi-square test of independence (Section A) |
| ANOVA with multiple factors + interactions | Logistic regression (Section B) |
| ANOVA's F-test ("which factor matters") | Analysis of Deviance — likelihood-ratio tests (Section C) |
| Paired t-test | McNemar's exact test (already used in `comparisons.csv`) |
| Box plot of a measurement | Bar chart + confidence interval on a proportion (Section E) |

Section F adds one genuinely continuous measurement — answer length — where box plots
*do* apply directly, as a complementary (not safety-equivalent) angle.

**Scope, same as the rest of this project's notebooks:** `analysis_eligible`,
generation+scoring both `completed`, `split != "heldout"` (heldout is reserved, not for
shaping conclusions), and `kind == "harmful"` for the safety outcome (`unsafe`); twins
are pulled back in only where noted. Every test here is on `unsafe`/`refused` unless
stated otherwise — nothing here re-derives `attack_success`/`over_refusal`, since those
columns are unpopulated in this data (see `load_combined_results.ipynb`).


## 0. Load data

In [ ]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "chat.py").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

# Set this to force a specific local folder (skips all Drive detection below).
LOCAL_DIR = None

# Where dataset2_2part_output_combiner.ipynb actually writes this, relative to "My Drive".
DRIVE_RELATIVE = Path("ez-penta/dataset2_eval_gguf/combined")
DRIVE_CANDIDATES = [
    Path("/content/drive/MyDrive") / DRIVE_RELATIVE,  # Colab, once mounted
    *[Path(p) / "My Drive" / DRIVE_RELATIVE
      for p in glob.glob(str(Path.home() / "Library/CloudStorage/GoogleDrive-*"))],  # Google Drive for Desktop (macOS)
    Path.home() / "Google Drive" / "My Drive" / DRIVE_RELATIVE,  # older Drive File Stream layout
]

if LOCAL_DIR:
    COMBINED_DIR = (Path.cwd() / LOCAL_DIR) if not Path(LOCAL_DIR).is_absolute() else Path(LOCAL_DIR)
else:
    COMBINED_DIR = next((p for p in DRIVE_CANDIDATES if p.exists()), None)
    if COMBINED_DIR is None and Path("/content").exists():
        from google.colab import drive
        drive.mount("/content/drive")
        COMBINED_DIR = Path("/content/drive/MyDrive") / DRIVE_RELATIVE
    if COMBINED_DIR is None:
        raise FileNotFoundError(
            "Could not find the Drive-synced `combined` folder. Checked:\n"
            + "\n".join(f"  {p}" for p in DRIVE_CANDIDATES)
            + "\nInstall/sign in to Google Drive for Desktop, run this notebook in Colab, "
              "or set LOCAL_DIR above to a local copy."
        )

if not (COMBINED_DIR / "observations.csv").exists():
    raise FileNotFoundError(f"{COMBINED_DIR / 'observations.csv'} not found.")
print(f"Reading combined results from: {COMBINED_DIR}")

observations = pd.read_csv(COMBINED_DIR / "observations.csv", low_memory=False)

usable = observations[
    (observations["analysis_eligible"] == True)
    & (observations["generation_status"] == "completed")
    & (observations["scoring_status"] == "completed")
    & (observations["split"] != "heldout")
].copy()

harmful = usable[usable["kind"] == "harmful"].copy()
harmful["unsafe"] = harmful["safety"].eq("Unsafe").astype(int)
harmful["refused"] = harmful["refused"].eq(1.0).astype(int)
for col in ["culture", "framing", "prompt_language", "category"]:
    harmful[col] = harmful[col].astype("category")

print(f"{len(harmful)} usable harmful-seed rows")
harmful[["culture", "framing", "prompt_language", "category", "unsafe", "refused"]].head()


## 1. Install `scipy` and `statsmodels`

Neither is in `requirements.txt`; both are lightweight, CPU-only, and don't conflict with anything pinned there. Skipped if already present.

In [ ]:

import importlib
import subprocess
import sys

for package in ["scipy", "statsmodels"]:
    try:
        importlib.import_module(package)
        print(f"{package}: already installed")
    except ImportError:
        print(f"{package}: installing...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
        print(f"{package}: installed")

import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests


## Section A: Chi-square tests of independence (univariate, one factor at a time)

For each factor (`culture`, `framing`, `prompt_language`) on its own: does it associate
with `unsafe`? With `refused`? This is the direct binary-outcome analog of a one-way
ANOVA — it tests whether the factor's levels have different rates, without controlling
for the other two factors (that's what Section B/C add). p-values are Holm-corrected
across all 6 tests (3 factors x 2 outcomes) since they're reported together.

In [ ]:

chi2_rows = []
for factor in ["culture", "framing", "prompt_language"]:
    for outcome in ["unsafe", "refused"]:
        table = pd.crosstab(harmful[factor], harmful[outcome])
        chi2, p, dof, _ = stats.chi2_contingency(table)
        chi2_rows.append({"factor": factor, "outcome": outcome, "chi2": chi2, "dof": dof,
                           "p_raw": p, "n": int(table.values.sum())})

chi2_table = pd.DataFrame(chi2_rows)
chi2_table["p_holm"] = multipletests(chi2_table["p_raw"], method="holm")[1]
chi2_table = chi2_table.round({"chi2": 2, "p_raw": 5, "p_holm": 5})
chi2_table


## Section B: Logistic regression (controls for all three factors at once)

The multi-factor analog of ANOVA. `unsafe ~ culture + framing + prompt_language` fits
all three simultaneously, so each coefficient is "the effect of this factor, holding the
other two constant" — unlike Section A's univariate tests. Baseline (reference) levels:
`culture=neutral`, `framing=control`, `prompt_language=en`. Coefficients are in
log-odds; Section D converts them to odds ratios for a more readable plot.

In [ ]:

model_main = smf.logit(
    "unsafe ~ C(culture, Treatment('neutral')) + C(framing, Treatment('control')) "
    "+ C(prompt_language, Treatment('en'))",
    data=harmful,
).fit(disp=0)
print(model_main.summary())


## Section C: Analysis of Deviance — which factor matters, and do interactions matter?

The binary-outcome analog of an ANOVA table: drop one factor at a time from the main-
effects model and compare deviance via a likelihood-ratio test (chi-square). A large,
significant drop means that factor matters even after controlling for the other two.

Then: does adding the `culture x framing` interaction improve the fit significantly?
This directly tests the earlier finding that `culture=us` combined with `framing=rp` is
worse than either alone would predict — i.e. whether that's a real interaction or just
two additive main effects that happened to stack.

In [ ]:
def lr_test(reduced, full):
    lr_stat = 2 * (full.llf - reduced.llf)
    df_diff = full.df_model - reduced.df_model
    p = stats.chi2.sf(lr_stat, df_diff)
    return lr_stat, df_diff, p


FORMULAS = {
    "full (culture + framing + language)":
        "unsafe ~ C(culture, Treatment('neutral')) + C(framing, Treatment('control')) "
        "+ C(prompt_language, Treatment('en'))",
    "drop culture": "unsafe ~ C(framing, Treatment('control')) + C(prompt_language, Treatment('en'))",
    "drop framing": "unsafe ~ C(culture, Treatment('neutral')) + C(prompt_language, Treatment('en'))",
    "drop language": "unsafe ~ C(culture, Treatment('neutral')) + C(framing, Treatment('control'))",
}
fitted = {name: smf.logit(formula, data=harmful).fit(disp=0) for name, formula in FORMULAS.items()}

deviance_rows = []
full = fitted["full (culture + framing + language)"]
for name in ["drop culture", "drop framing", "drop language"]:
    lr_stat, df_diff, p = lr_test(fitted[name], full)
    deviance_rows.append({"term_removed": name.removeprefix("drop "), "lr_chi2": lr_stat, "df": df_diff, "p": p})

deviance_table = pd.DataFrame(deviance_rows).round({"lr_chi2": 2, "p": 6})
print("Analysis of Deviance (Type II: each factor's contribution controlling for the other two)")
display(deviance_table)

# Interaction tests, kept in a named table (not just printed) so the Findings section
# at the end can reference them without relying on loop variables that get overwritten.
model_cf_interaction = smf.logit(
    "unsafe ~ C(culture, Treatment('neutral')) * C(framing, Treatment('control')) "
    "+ C(prompt_language, Treatment('en'))",
    data=harmful,
).fit(disp=0)
model_all_2way = smf.logit(
    "unsafe ~ C(culture, Treatment('neutral')) * C(framing, Treatment('control')) "
    "+ C(culture, Treatment('neutral')) * C(prompt_language, Treatment('en')) "
    "+ C(framing, Treatment('control')) * C(prompt_language, Treatment('en'))",
    data=harmful,
).fit(disp=0)

interaction_rows = []
for name, model in [("culture x framing", model_cf_interaction), ("all three two-way (block)", model_all_2way)]:
    lr_stat, df_diff, p = lr_test(full, model)
    interaction_rows.append({"interaction": name, "lr_chi2": lr_stat, "df": df_diff, "p": p})
interaction_table = pd.DataFrame(interaction_rows).round({"lr_chi2": 2, "p": 6})
print("\nInteraction tests (does adding this interaction improve the main-effects model?)")
display(interaction_table)


## Robustness check: does controlling for `category` change anything?

`category` is a strong confound (earlier: 34.2% unsafe for `sexual_content` vs 4.3% for
`self_harm_mental_health`) that wasn't in the model above. Refit the main-effects model
with `category` added and compare the culture/framing/language coefficients — if they
barely move, the Section B/C conclusions aren't just an artifact of category mix.

In [ ]:

model_with_category = smf.logit(
    "unsafe ~ C(culture, Treatment('neutral')) + C(framing, Treatment('control')) "
    "+ C(prompt_language, Treatment('en')) + C(category)",
    data=harmful,
).fit(disp=0)

compare = pd.DataFrame({
    "without_category": model_main.params, "with_category": model_with_category.params,
}).dropna()
compare["difference"] = (compare["with_category"] - compare["without_category"]).round(3)
compare.round(3)


## Section D: Forest plot — odds ratios from the main-effects model

Each row is one factor level vs. its baseline (`neutral` / `control` / `en`). A dot to
the right of the OR=1 line means that level raises the odds of an unsafe response
relative to baseline; the whiskers are the 95% confidence interval. This is the
standard way to visualize a fitted logistic regression, and reads at a glance which
effects are both large and well-supported (CI far from 1) vs. uncertain (CI crossing 1).

In [ ]:

import matplotlib.pyplot as plt

params = model_main.params.drop("Intercept")
conf = model_main.conf_int().drop("Intercept")
odds_ratios = pd.DataFrame({
    "label": params.index, "or": np.exp(params.values),
    "lo": np.exp(conf[0].values), "hi": np.exp(conf[1].values),
})


def clean_label(raw):
    # statsmodels formula labels look like C(culture, Treatment('neutral'))[T.us]
    factor = raw.split("C(")[1].split(",")[0]
    level = raw.split("[T.")[1].rstrip("]")
    return f"{factor} = {level}"


odds_ratios["label"] = odds_ratios["label"].map(clean_label)
odds_ratios = odds_ratios.sort_values("or")

fig, ax = plt.subplots(figsize=(7, 0.5 * len(odds_ratios) + 1))
y = np.arange(len(odds_ratios))
ax.errorbar(odds_ratios["or"], y, xerr=[odds_ratios["or"] - odds_ratios["lo"], odds_ratios["hi"] - odds_ratios["or"]],
            fmt="o", color="#2c3e50", ecolor="#7f8c8d", capsize=3)
ax.axvline(1, color="#c0392b", linestyle="--", linewidth=1, label="OR = 1 (no effect)")
ax.set_yticks(y)
ax.set_yticklabels(odds_ratios["label"])
ax.set_xlabel("Odds ratio (unsafe), vs. culture=neutral / framing=control / language=en")
ax.set_xscale("log")
ax.set_title("Forest plot: odds ratios from the main-effects logistic regression")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(COMBINED_DIR / "forest_plot_odds_ratios.png", dpi=130, bbox_inches="tight")
plt.show()


## Section E: Bar charts with 95% confidence intervals

The same `framing x culture x language` breakdown as `load_combined_results.ipynb`'s
no-category charts, but with a Wilson score interval on each bar — this is the binary-
data equivalent of showing a box plot's spread: it tells you whether a gap between bars
is large relative to its own uncertainty, which raw bar height alone doesn't.

In [ ]:

def wilson_interval(successes, n, z=1.959963984540054):
    if n == 0:
        return 0.0, 0.0
    p_hat = successes / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    half_width = (z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))) / denom
    return max(0.0, center - half_width) * 100, min(1.0, center + half_width) * 100


GROUP_COLS_NC = ["framing", "culture", "prompt_language"]
combo_nc = harmful.groupby(GROUP_COLS_NC, observed=True).agg(
    n=("unsafe", "size"), unsafe_count=("unsafe", "sum"), refused_count=("refused", "sum"),
).reset_index()
combo_nc = combo_nc.sort_values(GROUP_COLS_NC).reset_index(drop=True)
combo_nc["unsafe_pct"] = combo_nc["unsafe_count"] / combo_nc["n"] * 100
combo_nc["refused_pct"] = combo_nc["refused_count"] / combo_nc["n"] * 100
for metric, count_col in [("unsafe", "unsafe_count"), ("refused", "refused_count")]:
    bounds = combo_nc.apply(lambda row: wilson_interval(row[count_col], row["n"]), axis=1)
    combo_nc[f"{metric}_ci_lo"] = [b[0] for b in bounds]
    combo_nc[f"{metric}_ci_hi"] = [b[1] for b in bounds]

combo_nc["label"] = combo_nc[GROUP_COLS_NC].astype(str).agg(" | ".join, axis=1)
combo_nc.head()


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 0.3 * len(combo_nc) + 1), sharey=True)
y = np.arange(len(combo_nc))

for ax, metric, color in zip(axes, ["unsafe", "refused"], ["#c0392b", "#2980b9"]):
    pct = combo_nc[f"{metric}_pct"]
    lo, hi = combo_nc[f"{metric}_ci_lo"], combo_nc[f"{metric}_ci_hi"]
    ax.barh(y, pct, color=color, alpha=0.75)
    ax.errorbar(pct, y, xerr=[pct - lo, hi - pct], fmt="none", ecolor="black", capsize=2, linewidth=1)
    ax.set_xlabel(f"{metric} % (95% Wilson CI)")
    ax.set_xlim(0, 100)

axes[0].set_yticks(y)
axes[0].set_yticklabels(combo_nc["label"], fontsize=7)
fig.suptitle("unsafe % / refused % with 95% confidence intervals, by framing x culture x language")
fig.tight_layout()
fig.savefig(COMBINED_DIR / "pct_with_ci.png", dpi=130, bbox_inches="tight")
plt.show()


## Section F: Answer length by group (box plots) — a genuinely continuous variable

Not a safety measurement — a complementary check on verbosity. Does `framing` or
`culture` change how much the model writes, and does that correlate with refusal
(e.g. are refusals systematically short)? Character count of `answer`, boxed by
`culture`, `framing`, and `prompt_language` (3 side-by-side panels).

In [ ]:
harmful["answer_length"] = harmful["answer"].str.len()
answer_length_by_language = harmful.groupby("prompt_language", observed=True)["answer_length"].median()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, factor in zip(axes, ["culture", "framing", "prompt_language"]):
    groups = [g["answer_length"].values for _, g in harmful.groupby(factor, observed=True)]
    labels = [str(name) for name, _ in harmful.groupby(factor, observed=True)]
    ax.boxplot(groups, tick_labels=labels, showfliers=False)
    ax.set_title(f"Answer length by {factor}")
    ax.set_ylabel("characters")
    ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(COMBINED_DIR / "answer_length_boxplots.png", dpi=130, bbox_inches="tight")
plt.show()


In [ ]:
# Does refusal correlate with shorter answers? (within harmful-seed answers only)
answer_length_by_refusal = harmful.groupby("refused", observed=True)["answer_length"].describe()[["count", "mean", "50%", "std"]]
answer_length_by_refusal


## Key Findings

Generated live from the objects computed above (`chi2_table`, `model_main`,
`deviance_table`, `interaction_table`, `compare`, `answer_length_by_refusal`,
`answer_length_by_language`) — rerunning this notebook on updated data regenerates this
report rather than leaving stale numbers behind.


In [ ]:
print("=" * 78)
print("KEY FINDINGS")
print("=" * 78)

print(f"\n[A] Chi-square (univariate): all {len(chi2_table)} tests significant after Holm "
      f"correction (largest p_holm = {chi2_table['p_holm'].max():.5f}). culture, framing, "
      "and language are each associated with unsafe/refused on their own.")

print("\n[B] Logistic regression odds ratios, vs. culture=neutral / framing=control / language=en:")
or_table = pd.DataFrame({
    "odds_ratio": np.exp(model_main.params.drop("Intercept")),
    "p": model_main.pvalues.drop("Intercept"),
}).sort_values("odds_ratio", ascending=False)
for label, row in or_table.iterrows():
    verdict = "significant" if row["p"] < 0.05 else "not significant"
    print(f"    {label:45s} OR={row['odds_ratio']:.2f}  p={row['p']:.4f}  ({verdict})")

print("\n[C] Analysis of Deviance — each factor's contribution controlling for the other two:")
for _, row in deviance_table.sort_values("lr_chi2", ascending=False).iterrows():
    print(f"    {row['term_removed']:10s} chi2={row['lr_chi2']:6.2f}  df={int(row['df'])}  p={row['p']:.6f}")

print("\n    Interaction tests (does adding this interaction improve the main-effects model?):")
for _, row in interaction_table.iterrows():
    verdict = "significant" if row["p"] < 0.05 else "not significant"
    print(f"    {row['interaction']:28s} chi2={row['lr_chi2']:6.2f}  df={int(row['df'])}  "
          f"p={row['p']:.6f}  ({verdict})")

biggest_shift_label = compare["difference"].abs().idxmax()
biggest_shift = compare["difference"].abs().max()
print(f"\n[Robustness] Largest coefficient shift after controlling for category: "
      f"{biggest_shift:.3f} (on {biggest_shift_label}) — conclusions above hold up; "
      "not just a category-mix artifact.")

refused_mean = answer_length_by_refusal.loc[1, "mean"]
complied_mean = answer_length_by_refusal.loc[0, "mean"]
lang_medians = ", ".join(f"{lang}={length:.0f}" for lang, length in answer_length_by_language.items())
print(f"\n[F] Answer length: refused answers average {refused_mean:.0f} characters vs. "
      f"{complied_mean:.0f} for complied answers. Median length by language: {lang_medians}.")
print("    Caveat: Japanese text encodes more meaning per character than English, so the "
      "language gap above is partly a character-density artifact, not purely verbosity.")


## Methodology recap

- **Section A** (chi-square): each factor alone is associated with `unsafe`/`refused` —
  see `p_holm` for which survive multiple-comparison correction.
- **Section B/C** (logistic regression + Analysis of Deviance): tells you which factor
  matters *after controlling for the other two*, and whether `culture x framing`
  interacts (the `us x rp` finding) rather than just adding.
- **Robustness check**: whether those conclusions hold once `category` is controlled
  for.
- **Section D** (forest plot): the single clearest "what matters, how much, how
  confidently" visualization.
- **Section E** (CI bars): the binary-data equivalent of a box plot's spread, on the
  same framing x culture x language breakdown used elsewhere in this project.
- **Section F** (answer length): a different, continuous signal — genuine box plots,
  worth a look but not a safety measurement.
